# 04. Analyze

**Input:** `../data/interim/clean.pkl`, `../data/interim/raw.pkl`.
**Does:** the accountability gradient across actor types, the observed severity split by representation, an
ordered logit fit with court-clustered standard errors, and a scikit-learn severity classifier with
permutation importance and a confusion matrix.
**Output:** five figures in `../output/figures/`.

In [1]:
import pandas as pd, numpy as np
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.stats import norm
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier
np.random.seed(20)
plt.rcParams.update({"font.size":11,"axes.spines.top":False,"axes.spines.right":False})

clean = pd.read_pickle("../data/interim/clean.pkl")
raw   = pd.read_pickle("../data/interim/raw.pkl")

# ---------- functions (defined at top) ----------
def lever(o):
    if pd.isna(o): return "None recorded"
    s = str(o).lower()
    if any(k in s for k in ["bar referr","referral","suspen","disqualif"," dq","pro hac","censure",
        "grievance","disciplinary","revocation","revoked","admonish"]): return "Professional discipline"
    if any(k in s for k in ["monetary","fine","costs order","cost order","adverse cost","attorney fee",
        "fees and cost","$","damages","disgorge","penalty"]): return "Monetary"
    if any(k in s for k in ["struck","stricken","strike","dismiss","vacat","remand","reversed","set aside",
        "quash","withdrew","withdrawn","annul","retract","corrected","refile","amend","waived","default"]):
        return "Case/ruling remedy"
    if any(k in s for k in ["warning","caution","reprimand","rebuke","flag"]): return "Warning only"
    return "None recorded"

def actor_full(p):
    if pd.isna(p): return None
    if "Judge" in p: return "Judge"
    if p=="Pro Se Litigant": return "Pro se litigant"
    if p in {"Lawyer","Governement Lawyer","Prosecutor","Federal Defender"}: return "Lawyer (counseled)"
    return None

## (A) Accountability gradient (all actor types, full database)

In [2]:
raw["actor"]=raw["Party(ies)"].apply(actor_full); raw["lever"]=raw["Outcome"].apply(lever)
order=["Lawyer (counseled)","Pro se litigant","Judge"]
levels=["Professional discipline","Monetary","Case/ruling remedy","Warning only","None recorded"]
g=raw[raw["actor"].isin(order)]
tab=(g.groupby("actor")["lever"].value_counts(normalize=True).unstack()
       .reindex(index=order,columns=levels).fillna(0)*100)
n=g["actor"].value_counts().reindex(order)
print(tab.round(1).to_string())

cols={"Professional discipline":"#7b1f2b","Monetary":"#c0504d","Case/ruling remedy":"#e0a458",
      "Warning only":"#8ab6d6","None recorded":"#cccccc"}
fig,ax=plt.subplots(figsize=(9,4.8)); left=np.zeros(len(order))
for lv in levels:
    v=tab[lv].values; ax.barh(range(len(order)),v,left=left,color=cols[lv],label=lv,edgecolor="white"); left+=v
ax.set_yticks(range(len(order))); ax.set_yticklabels([f"{a}\n(n={int(n[a])})" for a in order]); ax.invert_yaxis()
ax.set_xlim(0,100); ax.set_xlabel("% of that actor's cases")
ax.set_title("Who bears the consequence for an AI hallucination?")
ax.legend(ncol=3,fontsize=8.5,frameon=False,loc="upper center",bbox_to_anchor=(0.5,-0.15))
plt.tight_layout(); plt.savefig("../output/figures/fig_accountability_gradient.png",dpi=200,bbox_inches="tight"); plt.close()
print("saved ../output/figures/fig_accountability_gradient.png")

lever               Professional discipline  Monetary  Case/ruling remedy  Warning only  None recorded
actor                                                                                                 
Lawyer (counseled)                     24.5      19.5                 5.3          13.2           37.6
Pro se litigant                         5.0       8.6                11.2          37.3           37.8
Judge                                  14.8       3.7                37.0           3.7           40.7
saved ../output/figures/fig_accountability_gradient.png


## (B) Observed severity by representation

In [3]:
d0=clean[clean["actor"].isin(["Pro se","Counseled"])]
dist=(d0.groupby("actor")["severity"].value_counts(normalize=True).unstack()
        .reindex(index=["Pro se","Counseled"],columns=range(5)).fillna(0)*100)
labs=["0 None","1 Warning","2 Procedural","3 Monetary","4 Prof/\nterminal"]; xg=np.arange(5); w=0.38
fig,ax=plt.subplots(figsize=(8,4.5))
ax.bar(xg-w/2,dist.loc["Pro se"].values,w,label=f"Pro se (n={int((clean.actor=='Pro se').sum())})",color="#4C72B0")
ax.bar(xg+w/2,dist.loc["Counseled"].values,w,label=f"Counseled (n={int((clean.actor=='Counseled').sum())})",color="#c0504d")
ax.set_xticks(xg); ax.set_xticklabels(labs,fontsize=9); ax.set_ylabel("% of cases")
ax.set_title("Sanction-severity distribution by representation"); ax.legend(frameon=False)
plt.tight_layout(); plt.savefig("../output/figures/fig_severity_dist.png",dpi=200); plt.close()
print("saved ../output/figures/fig_severity_dist.png")

saved ../output/figures/fig_severity_dist.png


## (C) Ordered logit (proportional odds), court-clustered SEs

In [4]:
d = clean[clean["actor"].isin(["Pro se","Counseled"])].dropna(subset=["year"]).copy()
d["pro_se"]=(d["actor"]=="Pro se").astype(int); d["year_c"]=d["year"]-2025
fd=pd.get_dummies(d["field"],prefix="f",drop_first=True).astype(float).reset_index(drop=True)
Xcols=["pro_se","federal","year_c"]+list(fd.columns)
X=pd.concat([d[["pro_se","federal","year_c"]].astype(float).reset_index(drop=True),fd],axis=1).values
y=d["severity"].values.astype(int); court=d["Court"].values; p=X.shape[1]

def Sg_(z): return 1/(1+np.exp(-z))
def cuts(th):
    c=np.empty(4); c[0]=th[0]
    for k in range(1,4): c[k]=c[k-1]+np.exp(th[k])
    return c
def ll_obs(par):
    b=par[:p]; c=cuts(par[p:p+4]); eta=X@b
    lo=np.where(y==0,-np.inf,np.take(c,np.clip(y-1,0,3)))
    hi=np.where(y==4, np.inf,np.take(c,np.clip(y,0,3)))
    return np.log(np.clip(np.where(np.isinf(hi),1,Sg_(hi-eta))-np.where(np.isinf(lo),0,Sg_(lo-eta)),1e-12,1))
negll=lambda par:-ll_obs(par).sum()
r=minimize(negll,np.r_[np.zeros(p),[-1,0,0,0]],method="Nelder-Mead",options={"maxiter":40000,"fatol":1e-8,"xatol":1e-8})
r=minimize(negll,r.x,method="BFGS",options={"maxiter":5000}); beta=r.x[:p]

def pg(par,eps=1e-5):
    G=np.zeros((len(y),len(par)))
    for j in range(len(par)):
        a=par.copy();a[j]+=eps;b=par.copy();b[j]-=eps;G[:,j]=(ll_obs(a)-ll_obs(b))/(2*eps)
    return G
def hess(par,eps=1e-4):
    nP=len(par);H=np.zeros((nP,nP))
    for i in range(nP):
        for j in range(i,nP):
            a=par.copy();a[i]+=eps;a[j]+=eps;b=par.copy();b[i]+=eps;b[j]-=eps
            c=par.copy();c[i]-=eps;c[j]+=eps;e=par.copy();e[i]-=eps;e[j]-=eps
            H[i,j]=H[j,i]=(negll(a)-negll(b)-negll(c)+negll(e))/(4*eps*eps)
    return H
Sm=pg(r.x); Hi=np.linalg.pinv(hess(r.x)); meat=np.zeros((len(r.x),)*2)
for gg in np.unique(court):
    u=Sm[court==gg].sum(0); meat+=np.outer(u,u)
se=np.sqrt(np.diag(Hi@meat@Hi))[:p]
res=pd.DataFrame({"term":Xcols,"OR":np.exp(beta),"lo":np.exp(beta-1.96*se),
                  "hi":np.exp(beta+1.96*se),"p":2*(1-norm.cdf(np.abs(beta/se)))})
print(res.round(3).to_string(index=False))

show=["pro_se","federal","year_c"]+[c for c in Xcols if c.startswith("f_")]
sub=res.set_index("term").loc[show]; yy=np.arange(len(show))[::-1]
fig,ax=plt.subplots(figsize=(8,5))
ax.errorbar(sub["OR"],yy,xerr=[sub["OR"]-sub["lo"],sub["hi"]-sub["OR"]],fmt="o",color="#2f4b7c",capsize=3)
ax.axvline(1,color="#999",ls="--",lw=1); ax.set_yticks(yy); ax.set_yticklabels(show,fontsize=9)
ax.set_xscale("log"); ax.set_xlabel("Odds ratio (higher severity), log scale")
ax.set_title("Ordered-logit odds ratios, 95% CI (court-clustered)")
plt.tight_layout(); plt.savefig("../output/figures/fig_forest.png",dpi=200); plt.close()
print("saved ../output/figures/fig_forest.png")

          term    OR    lo    hi     p
        pro_se 0.323 0.253 0.412 0.000
       federal 1.224 0.991 1.512 0.061
        year_c 0.764 0.628 0.929 0.007
f_civil rights 1.050 0.671 1.641 0.832
    f_contract 1.050 0.659 1.676 0.836
  f_employment 0.843 0.532 1.335 0.467
      f_family 0.708 0.369 1.360 0.300
       f_other 1.210 0.767 1.908 0.413
        f_tort 0.830 0.493 1.398 0.484
saved ../output/figures/fig_forest.png


## (D) Supervised ML: predicting severity (scikit-learn)

In [5]:
feat=pd.concat([d[["pro_se","federal","year","tool_named"]].reset_index(drop=True),
                pd.get_dummies(d["field"],prefix="f").astype(int).reset_index(drop=True)],axis=1)
Xtr,Xte,ytr,yte=train_test_split(feat,y,test_size=0.25,stratify=y,random_state=20)
base=DummyClassifier(strategy="most_frequent").fit(Xtr,ytr)
clf=make_pipeline(StandardScaler(),LogisticRegression(max_iter=5000)).fit(Xtr,ytr)
pr=clf.predict(Xte)
print("baseline acc:",round((base.predict(Xte)==yte).mean(),3),
      "| model acc:",round((pr==yte).mean(),3),
      "| macro-F1:",round(f1_score(yte,pr,average='macro'),3))

pi=permutation_importance(clf,Xte,yte,n_repeats=30,random_state=20,scoring="f1_macro")
imp=pd.Series(pi.importances_mean,index=feat.columns).sort_values()
fig,ax=plt.subplots(figsize=(7,4.2)); ax.barh(range(len(imp)),imp.values,color="#2f4b7c")
ax.set_yticks(range(len(imp))); ax.set_yticklabels(imp.index,fontsize=9)
ax.set_xlabel("Permutation importance (macro-F1 drop)"); ax.set_title("What predicts sanction severity?")
plt.tight_layout(); plt.savefig("../output/figures/fig_ml_importance.png",dpi=200); plt.close()

cm=confusion_matrix(yte,pr,normalize="true")
labs2=["None","Warning","Procedural","Monetary","Prof/term"]
fig,ax=plt.subplots(figsize=(5.5,4.8)); im=ax.imshow(cm,cmap="Blues",vmin=0,vmax=1)
ax.set_xticks(range(5)); ax.set_yticks(range(5)); ax.set_xticklabels(labs2,rotation=40,ha="right"); ax.set_yticklabels(labs2)
for i in range(5):
    for j in range(5): ax.text(j,i,f"{cm[i,j]:.2f}",ha="center",va="center",fontsize=8,color="white" if cm[i,j]>.5 else "#333")
ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title("Confusion matrix (row-normalized)")
plt.colorbar(im,fraction=0.046); plt.tight_layout(); plt.savefig("../output/figures/fig_ml_confusion.png",dpi=200); plt.close()
print("saved ../output/figures/fig_ml_importance.png and ../output/figures/fig_ml_confusion.png")

baseline acc: 0.394 | model acc: 0.422 | macro-F1: 0.254
saved ../output/figures/fig_ml_importance.png and ../output/figures/fig_ml_confusion.png


## (E) Pooled AI vs non-AI comparison

The reviewer's question: do AI-hallucination filings draw a different sanction response than
*comparable non-AI* sanctions cases? This section pools the AI arm with the CourtListener control
group (`../data/coded/pooled_coded.csv`) and regresses severity on an `ai` dummy plus controls.

**Read the data-quality guard below before trusting anything here.** The control severity is only
valid if `02_build_controls` was run with the current `label_lib` (strict full-opinion coder +
bar-discipline filter). If the guard fires, re-run `02` before reporting the `ai` coefficient.

In [6]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

pooled = pd.read_csv("../data/coded/pooled_coded.csv")
print("arms:", pooled["ai"].value_counts().to_dict())
by = pooled.groupby("ai")["severity"].value_counts().unstack().fillna(0).astype(int)
print("severity by arm (0=control, 1=AI):\n", by.to_string())

# ---- data-quality guard ----
ctrl = pooled[pooled.ai==0]["severity"]
t4_share = (ctrl==4).mean(); warn_share = (ctrl==1).mean()
provisional = (t4_share > 0.25) or (warn_share < 0.05)
if provisional:
    print("\n*** WARNING: control severity looks over-coded "
          f"(tier-4 share={t4_share:.0%}, warning share={warn_share:.0%}). ***")
    print("*** This means controls_coded.csv was built with the OLD coder. Re-run")
    print("*** 02_build_controls with the current label_lib, then re-run this cell.")
    print("*** Results below are PROVISIONAL and must not be reported as-is.\n")

# ---- mean severity by arm ----
m = pooled.groupby("ai")["severity"].agg(["size","mean"])
fig,ax=plt.subplots(figsize=(5,4))
ax.bar(["Control\n(non-AI)","AI"], m["mean"].values, color=["#8ab6d6","#c0504d"])
for i,v in enumerate(m["mean"].values): ax.text(i,v,f"{v:.2f}",ha="center",va="bottom")
ax.set_ylabel("Mean severity (0-4)"); ax.set_title("Mean sanction severity by arm"+("  [PROVISIONAL]" if provisional else ""))
plt.tight_layout(); plt.savefig("../output/figures/fig_pooled_means.png",dpi=200); plt.show()

# ---- pooled OLS: severity ~ ai + year_c + field dummies ----
d = pooled.dropna(subset=["severity","field","year"]).copy()
d["year_c"] = d["year"] - 2025
fd = pd.get_dummies(d["field"], prefix="f", drop_first=True).astype(float).reset_index(drop=True)
Xcols = ["ai","year_c"] + list(fd.columns)
X = np.column_stack([np.ones(len(d)), d[["ai","year_c"]].astype(float).reset_index(drop=True).values, fd.values])
y = d["severity"].astype(float).values
XtX_inv = np.linalg.pinv(X.T@X); b = XtX_inv@X.T@y; e = y - X@b
sigma2 = (e@e)/(len(y)-X.shape[1]); V = sigma2*XtX_inv; se = np.sqrt(np.diag(V))
names = ["intercept"]+Xcols
print("\nPooled OLS  severity ~ ai + year_c + field   (n=%d)"%len(d))
print(f"{'term':<16}{'beta':>8}{'SE':>7}{'p':>9}")
for i,nm in enumerate(names):
    p=2*(1-norm.cdf(abs(b[i]/se[i])))
    print(f"{nm:<16}{b[i]:>8.3f}{se[i]:>7.3f}{p:>9.4f}")
ai_i = names.index("ai")
print(f"\nAI coefficient: {b[ai_i]:+.3f} severity tiers "
      f"(95% CI {b[ai_i]-1.96*se[ai_i]:+.3f} to {b[ai_i]+1.96*se[ai_i]:+.3f})"
      + ("  [PROVISIONAL]" if provisional else ""))

# ---- beta forest (coefficients, reference line at 0) ----
show=[n for n in names if n!="intercept"]; idx=[names.index(s) for s in show]
yy=np.arange(len(show))[::-1]
fig,ax=plt.subplots(figsize=(8,4.6))
ax.errorbar(b[idx],yy,xerr=1.96*se[idx],fmt="o",color="#2f4b7c",capsize=3)
ax.axvline(0,color="#999",ls="--",lw=1); ax.set_yticks(yy); ax.set_yticklabels(show,fontsize=9)
ax.set_xlabel(r"OLS coefficient ($\beta$) on severity, 95% CI"); ax.set_title("Pooled model: what moves severity?"+("  [PROVISIONAL]" if provisional else ""))
plt.tight_layout(); plt.savefig("../output/figures/fig_pooled_forest.png",dpi=200); plt.show()

arms: {1: 1279, 0: 321}
severity by arm (0=control, 1=AI):
 severity    0    1    2    3    4
ai                               
0         203   15   20   28   55
1         256  498  217  135  173

*** WARNING: control severity looks over-coded (tier-4 share=17%, warning share=5%). ***
*** This means controls_coded.csv was built with the OLD coder. Re-run
*** 02_build_controls with the current label_lib, then re-run this cell.
*** Results below are PROVISIONAL and must not be reported as-is.


Pooled OLS  severity ~ ai + year_c + field   (n=1599)
term                beta     SE        p
intercept          0.888  0.180   0.0000
ai                 0.685  0.113   0.0000
year_c            -0.062  0.050   0.2128
f_civil rights    -0.026  0.178   0.8852
f_contract         0.077  0.165   0.6397
f_employment      -0.061  0.193   0.7532
f_family          -0.228  0.208   0.2750
f_other            0.202  0.164   0.2169
f_tort            -0.057  0.189   0.7645

AI coefficient: +0.685 severity tiers

/var/folders/z4/btgxcprj5g3dyfktkh04y9s00000gn/T/ipykernel_10281/3203664896.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("../output/figures/fig_pooled_means.png",dpi=200); plt.show()
/var/folders/z4/btgxcprj5g3dyfktkh04y9s00000gn/T/ipykernel_10281/3203664896.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig("../output/figures/fig_pooled_forest.png",dpi=200); plt.show()


### Caveats for the pooled comparison
- **Control frame.** Controls are non-AI cases where a court ruled on a sanctions / Rule 11 / §1927
  question, drawn from the same 2023–2026 window; standalone bar-discipline proceedings are excluded
  (`label_lib.is_bar_discipline`). This holds "a sanctionable issue was litigated" roughly constant.
- **Coding symmetry.** The AI arm is coded from Charlotin's short `Outcome` field; controls from full
  opinions with the strict imposed-sanction coder. Different sources coded as alike as possible, but
  not identical — validate a sample with `validate_coding.py`.
- **`pro_se` and `federal` are missing for most controls**, so they are left out of the pooled model.
- **Not causal.** AI use is not randomly assigned; the `ai` coefficient is an association.